# ToysAD-8K DS0 LocateAnything Proposals

File: `04_toysad8k_ds0_locateanything_proposals.ipynb`

On Kaggle with a GPU, this notebook runs the pinned LocateAnything model on the accepted opaque package from Notebook 03. The one-image and repeated 20-image gates have passed, so the current default runs all 400 images. Local execution uses a synthetic contract worker and does not run LocateAnything.

It does not read categories, source identifiers, roles, anomaly labels, reference boxes, or masks. Proposal quality is measured later by a separate evaluator-only stage.


## Acceptance contract

Local execution runs a deterministic 400-image contract smoke without loading the 3B model. This checks input verification, job construction, parsing, retry and resume behaviour, evidence tables, figures, and output hashes. Its boxes are synthetic and must not be used to judge proposal quality.

Kaggle execution is accepted only when the pinned model passes one-image and 20-image preflight checks, the 20-image repeat produces identical parsed boxes, all 400 jobs finish without failure or truncation, and the downloaded outputs and ClearML evidence pass independent verification.

A successful one-image or pilot run is recorded as `diagnostic_completed`. It confirms only that the requested gate worked and is not an accepted proposal dataset.

The project tests the released model before considering training or fine-tuning. Model adoption is decided only after a separate evaluator joins the frozen proposals to protected reference boxes and reports count accuracy, precision, recall, matched IoU, misses, extras, merges, splits, repeatability, latency, and failure cases.


In [ ]:
from hashlib import sha256
from html import escape
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import threading
import time

from IPython.display import HTML, display
from PIL import Image


In [ ]:
NOTEBOOK_NAME = "04_toysad8k_ds0_locateanything_proposals.ipynb"


def find_code_root(start: Path) -> Path:
    candidates = []
    configured = os.getenv("VPI_CODE_ROOT")
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend((start, *start.parents))
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for mounted in sorted(kaggle_input.iterdir()):
            candidates.extend((mounted, mounted / "code"))
        for marker in kaggle_input.rglob("locateanything.py"):
            if marker.parts[-2:] == (
                "models",
                "locateanything.py",
            ):
                candidates.append(marker.parents[3])
    for candidate in candidates:
        marker = candidate / "src" / "visual_peer_inspector" / "models" / "locateanything.py"
        if marker.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the code-only snapshot. Set VPI_CODE_ROOT to the directory containing src/visual_peer_inspector."
    )


def find_model_input_root(code_root: Path) -> Path:
    configured = os.getenv("VPI_MODEL_INPUT_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.append(code_root / ".local" / "toysad" / "runs" / "opaque-package-smoke20" / "model-input")
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for mounted in sorted(kaggle_input.iterdir()):
            candidates.extend((mounted, mounted / "model-input"))
        candidates.extend(
            path.parent for path in kaggle_input.rglob("package_record.json")
        )
    for candidate in candidates:
        if (
            (candidate / "neutral_corpus_manifest.jsonl").is_file()
            and (candidate / "package_record.json").is_file()
            and (candidate / "content_checksums.sha256").is_file()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the accepted Notebook 03 model-input package. Set VPI_MODEL_INPUT_ROOT."
    )


def prepare_clearml_tracking(execution_environment: str) -> str:
    default_mode = "online" if execution_environment == "kaggle" else "disabled"
    tracking_mode = os.getenv("VPI_TRACKING_MODE", default_mode).lower()
    os.environ["VPI_TRACKING_MODE"] = tracking_mode
    os.environ.setdefault("CLEARML_LOG_ENVIRONMENT", "")
    if tracking_mode == "disabled":
        return tracking_mode
    if importlib.util.find_spec("clearml") is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "clearml==2.1.10"]
        )
    if tracking_mode == "online" and execution_environment == "kaggle":
        from kaggle_secrets import UserSecretsClient

        secret_client = UserSecretsClient()
        for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
            try:
                value = secret_client.get_secret(name)
            except Exception as error:
                raise RuntimeError(f"Kaggle Secret {name} is missing or disabled.") from error
            if not value:
                raise RuntimeError(f"Kaggle Secret {name} is empty.")
            os.environ[name] = value
        os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
        os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
        os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    return tracking_mode


def verify_snapshot_record(code_root: Path, snapshot_path: Path) -> dict:
    from visual_peer_inspector.verification import verify_code_snapshot_record

    return verify_code_snapshot_record(code_root, snapshot_path)


def read_provenance(code_root: Path, notebook_name: str) -> dict:
    notebook_path = code_root / "notebooks" / notebook_name
    if not notebook_path.is_file():
        raise FileNotFoundError(f"Canonical notebook source is missing: {notebook_name}")
    snapshot_candidates = [
        Path(os.environ["VPI_SNAPSHOT_RECORD"]).expanduser()
        if os.getenv("VPI_SNAPSHOT_RECORD") else None,
        code_root / "snapshot_record.json",
        code_root / ".local" / "kaggle_code_snapshot" / "snapshot_record.json",
    ]
    snapshot_path = next(
        (path for path in snapshot_candidates if path is not None and path.is_file()),
        None,
    )
    if snapshot_path is not None:
        snapshot = verify_snapshot_record(code_root, snapshot_path)
        git_commit = snapshot["git_commit"]
        git_dirty = bool(snapshot["git_dirty"])
        snapshot_sha256 = snapshot["snapshot_sha256"]
        platform_versions = dict(snapshot.get("platform_versions", {}))
        precommit_candidate = bool(snapshot.get("precommit_candidate", False))
    else:
        repository_root = code_root.parent
        git_commit = subprocess.check_output(
            ["git", "-C", str(repository_root), "rev-parse", "HEAD"],
            text=True,
        ).strip()
        git_dirty = bool(
            subprocess.check_output(
                ["git", "-C", str(repository_root), "status", "--porcelain"],
                text=True,
            ).strip()
        )
        snapshot_sha256 = "local-working-tree"
        platform_versions = {}
        precommit_candidate = git_dirty
    return {
        "git_commit": git_commit,
        "git_dirty": git_dirty,
        "snapshot_sha256": snapshot_sha256,
        "precommit_candidate": precommit_candidate,
        "platform_versions": platform_versions,
        "notebook_sha256": sha256(notebook_path.read_bytes()).hexdigest(),
    }


def display_rows(rows: list[dict], columns: list[str], limit: int | None = None) -> None:
    shown = rows if limit is None else rows[:limit]
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns) + "</tr>"
        for row in shown
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


## Runtime, package, and GPU preflight

Kaggle must attach only the dedicated code-only snapshot and the accepted Notebook 03 model-input dataset. The selection-audit dataset, trusted labels, source maps, and evaluator files must not be attached.


In [ ]:
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/working").is_dir() else "local"
RUN_MODE = "model_inference" if EXECUTION_ENVIRONMENT == "kaggle" else "contract_smoke"
default_inference_stage = "full"
INFERENCE_STAGE = os.getenv("VPI_INFERENCE_STAGE", default_inference_stage).strip().lower()
if INFERENCE_STAGE not in {"one_image", "pilot", "full"}:
    raise ValueError("VPI_INFERENCE_STAGE must be one_image, pilot, or full.")
CODE_ROOT = find_code_root(Path.cwd().resolve())
sys.path.insert(0, str(CODE_ROOT / "src"))

if RUN_MODE == "model_inference":
    requirement = CODE_ROOT / "requirements" / "locateanything-lock.txt"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirement)])

TRACKING_MODE = prepare_clearml_tracking(EXECUTION_ENVIRONMENT)

from visual_peer_inspector.data.model_safe_package import (
    SMOKE20_COHORT_ID,
    canonical_json_bytes,
    file_sha256,
    verify_model_input_package,
)
from visual_peer_inspector.evidence import (
    save_bar_chart,
    save_histogram,
    save_image_grid,
    save_scatter_plot,
    write_csv_rows,
)
from visual_peer_inspector.models.locateanything import (
    CONTRACT_PRODUCER_KIND,
    LocateAnythingConfiguration,
    LocateAnythingWorker,
    MODEL_PRODUCER_KIND,
    ProposalContractWorker,
    ProposalResultStore,
    build_proposal_jobs,
    collect_proposal_evidence,
    repeat_agreement,
    run_proposal_jobs,
    summarize_proposal_rows,
)
from visual_peer_inspector.tracking import ExperimentTracker

MODEL_INPUT_ROOT = find_model_input_root(CODE_ROOT)
package_audit = verify_model_input_package(MODEL_INPUT_ROOT)
default_output = (
    Path("/kaggle/working/toysad8k_ds0_locateanything_proposals")
    if RUN_MODE == "model_inference"
    else CODE_ROOT / ".local" / "toysad" / "runs" / "locateanything-contract-smoke"
)
OUTPUT_ROOT = Path(os.getenv("VPI_OUTPUT_ROOT", str(default_output))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT = OUTPUT_ROOT / "report"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
PROVENANCE = read_provenance(CODE_ROOT, NOTEBOOK_NAME)
if (
    EXECUTION_ENVIRONMENT == "kaggle"
    and PROVENANCE["git_dirty"]
    and not PROVENANCE["precommit_candidate"]
):
    raise RuntimeError(
        "A dirty Kaggle snapshot must be marked as a pre-commit candidate."
    )
version_variables = {
    "kaggle_notebook_version_04": "VPI_KAGGLE_NOTEBOOK_VERSION_04",
    "code_dataset_version": "VPI_CODE_DATASET_VERSION",
    "model_input_dataset_version": "VPI_MODEL_INPUT_DATASET_VERSION",
}
PLATFORM_VERSIONS = {
    key: os.getenv(variable) or PROVENANCE["platform_versions"].get(key)
    for key, variable in version_variables.items()
}
missing_versions = [key for key, value in PLATFORM_VERSIONS.items() if not value]
if EXECUTION_ENVIRONMENT == "kaggle" and missing_versions:
    raise RuntimeError(
        f"Kaggle version metadata is missing: {missing_versions}"
    )
PLATFORM_VERSIONS = {
    key: value or "local" for key, value in PLATFORM_VERSIONS.items()
}

if RUN_MODE == "model_inference":
    import torch
    import transformers

    if not torch.cuda.is_available():
        raise RuntimeError("A Kaggle GPU is required for LocateAnything model inference.")
    gpu_properties = torch.cuda.get_device_properties(0)
    current_architecture = f"sm_{gpu_properties.major}{gpu_properties.minor}"
    supported_architectures = set(torch.cuda.get_arch_list())
    torch_architecture_supported = (
        not supported_architectures
        or current_architecture in supported_architectures
    )
    if not torch_architecture_supported:
        raise RuntimeError(
            f"The selected GPU uses {current_architecture}, but the installed "
            f"PyTorch build supports {sorted(supported_architectures)}. "
            "Select a compatible Kaggle accelerator before running inference."
        )
    gpu_evidence = {
        "gpu_name": torch.cuda.get_device_name(0),
        "compute_capability": f"{gpu_properties.major}.{gpu_properties.minor}",
        "total_memory_gib": round(gpu_properties.total_memory / (1024 ** 3), 3),
        "torch_architecture": current_architecture,
        "torch_architecture_supported": torch_architecture_supported,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    }
else:
    torch = None
    transformers = None
    gpu_evidence = {
        "gpu_name": "not used",
        "compute_capability": "not applicable",
        "total_memory_gib": 0,
        "torch_architecture": "not applicable",
        "torch_architecture_supported": "not applicable",
        "torch_version": "not loaded",
        "transformers_version": "not loaded",
    }

print(f"Run mode: {RUN_MODE}")
print(f"Inference stage: {INFERENCE_STAGE}")
print(f"Verified model input: {package_audit.scene_count} scenes, {package_audit.image_count} images")
print(f"Output folder name: {OUTPUT_ROOT.name}")
print(f"Tracking mode: {TRACKING_MODE}")
display_rows(
    [{"measure": key, "value": value} for key, value in gpu_evidence.items()],
    ["measure", "value"],
)
display_rows(
    [{"resource": key, "version": value} for key, value in PLATFORM_VERSIONS.items()],
    ["resource", "version"],
)


## Pinned configuration and resumable job list

The model, revision, prompt, generation settings, input manifest, notebook, code snapshot, and environment are fixed before execution. Existing results are reused only when their configuration, cohort, input manifest, and image hashes match the current jobs.


In [ ]:
configuration = LocateAnythingConfiguration()
producer_kind = (
    MODEL_PRODUCER_KIND
    if RUN_MODE == "model_inference"
    else CONTRACT_PRODUCER_KIND
)
jobs = build_proposal_jobs(MODEL_INPUT_ROOT, SMOKE20_COHORT_ID)
if len(jobs) != 400:
    raise ValueError(f"DS0 requires 400 all-view jobs, found {len(jobs)}")
result_store = ProposalResultStore(
    OUTPUT_ROOT / "proposal_results.jsonl",
    configuration.sha256,
    producer_kind,
)
result_store.reconcile_jobs(jobs)

tracking_configuration = {
    "run_mode": RUN_MODE,
    "inference_stage": INFERENCE_STAGE,
    "producer_kind": producer_kind,
    "model_id": configuration.model_id,
    "model_revision": configuration.model_revision,
    "prompt": configuration.prompt,
    "upscale_factor": configuration.upscale_factor,
    "seed": configuration.seed,
    "max_new_tokens": configuration.max_new_tokens,
    "generation_mode": configuration.generation_mode,
    "do_sample": configuration.do_sample,
    "temperature": configuration.temperature,
    "top_p": configuration.top_p,
    "top_k": configuration.top_k,
    "repetition_penalty": configuration.repetition_penalty,
    "scene_count": package_audit.scene_count,
    "views_per_scene": package_audit.view_count,
    "maximum_attempts_per_job": 2,
    "maximum_consecutive_failed_jobs": 3,
}
tracking_properties = {
    "execution_environment": EXECUTION_ENVIRONMENT,
    "run_mode": RUN_MODE,
    "inference_stage": INFERENCE_STAGE,
    "producer_kind": producer_kind,
    "cohort_id": SMOKE20_COHORT_ID,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_sha256": PROVENANCE["notebook_sha256"],
    "git_commit": PROVENANCE["git_commit"],
    "git_dirty": PROVENANCE["git_dirty"],
    "snapshot_sha256": PROVENANCE["snapshot_sha256"],
    "kaggle_notebook_slug": "esreekarreddy/toysad-8k-ds0-locateanything-proposals",
    "input_manifest_sha256": package_audit.manifest_sha256,
    "configuration_sha256": configuration.sha256,
    "model_id": configuration.model_id,
    "model_revision": configuration.model_revision,
    "python_version": platform.python_version(),
    **PLATFORM_VERSIONS,
    **gpu_evidence,
}
tracker = ExperimentTracker(mode="disabled")
print(f"Jobs: {len(jobs)} total, {len(result_store.completed_keys)} successful, {len(jobs) - len(result_store.completed_keys)} pending")
display_rows(
    [{"setting": key, "value": value} for key, value in tracking_configuration.items()],
    ["setting", "value"],
)


## One-image and 20-image preflight

The one-image recovery gate and the repeated 20-image pilot have passed. The full run still repeats these checks before processing the remaining views. Model download and load checkpoints are saved so a stalled run can be diagnosed without waiting for the Kaggle time limit.


In [ ]:
run_error = None
model_load_seconds = 0.0
model_download_seconds = 0.0
progress_rows = []
model_load_rows = []
repeat_summary = {"job_count": 0, "matching_job_count": 0, "box_agreement_rate": 0.0}
one_image_summary = {}
pilot_summary = {}
run_session = {}
estimated_full_run_seconds = 0.0
repeat_results_path = OUTPUT_ROOT / "repeat_pilot_results.jsonl"
model_load_path = REPORT_ROOT / "model_load_checkpoints.jsonl"
model_load_path.unlink(missing_ok=True)
load_lock = threading.Lock()
load_started = time.perf_counter()


def record_model_load(stage: str, details: dict | None = None) -> None:
    details = dict(details or {})
    disk_root = "/kaggle/working" if EXECUTION_ENVIRONMENT == "kaggle" else OUTPUT_ROOT
    disk = shutil.disk_usage(disk_root)
    row = {
        "stage": stage,
        "elapsed_seconds": round(time.perf_counter() - load_started, 3),
        "working_free_gib": round(disk.free / (1024 ** 3), 3),
        **details,
    }
    if RUN_MODE == "model_inference":
        row["gpu_allocated_gib"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        row["gpu_reserved_gib"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    with load_lock:
        model_load_rows.append(row)
        with model_load_path.open("a", encoding="utf-8", newline="\n") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
            handle.flush()
    detail_text = ", ".join(
        f"{key}={value}" for key, value in row.items() if key != "stage"
    )
    print(f"Model checkpoint: {stage} ({detail_text})", flush=True)


load_monitor_stop = threading.Event()


def report_load_wait() -> None:
    while not load_monitor_stop.wait(60):
        record_model_load("waiting")

try:
    load_monitor = threading.Thread(target=report_load_wait, daemon=True)
    load_monitor.start()
    try:
        if RUN_MODE == "model_inference":
            from huggingface_hub import snapshot_download

            record_model_load("snapshot_download_started")
            download_started = time.perf_counter()
            model_source = snapshot_download(
                repo_id=configuration.model_id,
                revision=configuration.model_revision,
                cache_dir="/root/.cache/huggingface/hub",
                max_workers=4,
            )
            model_download_seconds = time.perf_counter() - download_started
            snapshot_files = [
                path for path in Path(model_source).rglob("*") if path.is_file()
            ]
            record_model_load(
                "snapshot_download_completed",
                {
                    "download_seconds": round(model_download_seconds, 3),
                    "snapshot_file_count": len(snapshot_files),
                    "snapshot_gib": round(
                        sum(path.stat().st_size for path in snapshot_files) / (1024 ** 3),
                        3,
                    ),
                },
            )
            worker = LocateAnythingWorker(
                configuration,
                model_source=model_source,
                progress_callback=record_model_load,
            )
        else:
            record_model_load("contract_worker_started")
            worker = ProposalContractWorker(configuration)
            record_model_load("contract_worker_ready")
    finally:
        load_monitor_stop.set()
        load_monitor.join(timeout=2)
    model_load_seconds = time.perf_counter() - load_started
    record_model_load(
        "worker_ready",
        {"model_load_seconds": round(model_load_seconds, 3)},
    )
    if RUN_MODE == "model_inference":
        torch.cuda.reset_peak_memory_stats()

    pilot_jobs = [job for job in jobs if job.view_id == "V00"]
    if len(pilot_jobs) != 20:
        raise ValueError("The fixed preflight must contain one V00 image per scene.")

    def record_progress(completed: int, total: int, row: dict) -> None:
        overall_completed = len(result_store.completed_keys)
        progress = {
            "session_completed": completed,
            "session_total": total,
            "overall_successful": overall_completed,
            "latest_status": row["status"],
        }
        progress_rows.append(progress)
        if completed == 1 or completed % 20 == 0 or completed == total:
            tracker.report_scalar(
                title="DS0 progress",
                series="successful jobs",
                value=overall_completed,
                iteration=overall_completed,
            )
            print(f"{completed:>3}/{total} session jobs, {overall_completed:>3}/400 successful overall")

    run_proposal_jobs(
        worker=worker,
        jobs=[pilot_jobs[0]],
        result_store=result_store,
        maximum_attempts_per_job=2,
        maximum_consecutive_failed_jobs=1,
        progress_callback=record_progress,
    )
    one_image_row = next(
        row for row in result_store.rows
        if (row["opaque_scene_id"], row["view_id"]) == pilot_jobs[0].key
    )
    if one_image_row["status"] != "ok" or one_image_row["proposal_count"] < 1:
        raise RuntimeError("The one-image model response did not contain a valid proposal.")
    one_image_summary = {
        "status": one_image_row["status"],
        "proposal_count": one_image_row["proposal_count"],
        "response_token_count": one_image_row["response_token_count"],
        "possibly_truncated": one_image_row["possibly_truncated"],
        "truncation_check": one_image_row["truncation_check"],
        "result_sha256": sha256(canonical_json_bytes(one_image_row)).hexdigest(),
    }

    if INFERENCE_STAGE in {"pilot", "full"}:
        run_proposal_jobs(
            worker=worker,
            jobs=pilot_jobs,
            result_store=result_store,
            maximum_attempts_per_job=2,
            maximum_consecutive_failed_jobs=1,
            progress_callback=record_progress,
        )
        first_pilot_rows = [
            row for row in result_store.rows if row["view_id"] == "V00"
        ]
        if len(first_pilot_rows) != 20:
            raise RuntimeError("The first 20-image pilot did not complete.")
        pilot_summary = summarize_proposal_rows(first_pilot_rows)
        estimated_full_run_seconds = (
            model_load_seconds
            + pilot_summary["mean_runtime_seconds"] * len(jobs)
        )
        maximum_estimated_seconds = float(
            os.getenv("VPI_MAX_ESTIMATED_RUN_SECONDS", "36000")
        )
        if (
            RUN_MODE == "model_inference"
            and estimated_full_run_seconds > maximum_estimated_seconds
        ):
            raise RuntimeError(
                "The 20-image pilot projects beyond the configured Kaggle runtime budget."
            )
        second_pilot_rows = [worker.run_job(job, attempt=1) for job in pilot_jobs]
        if any(
            row["status"] != "ok" or row["proposal_count"] < 1
            for row in second_pilot_rows
        ):
            raise RuntimeError("The repeated 20-image pilot contains an invalid proposal result.")
        repeat_payload = b"".join(
            canonical_json_bytes(row) + b"\n" for row in second_pilot_rows
        )
        repeat_temporary = repeat_results_path.with_suffix(".jsonl.incomplete")
        repeat_temporary.write_bytes(repeat_payload)
        os.replace(repeat_temporary, repeat_results_path)
        repeat_summary = repeat_agreement(first_pilot_rows, second_pilot_rows)
        repeat_summary["result_file_sha256"] = file_sha256(repeat_results_path)
        if repeat_summary["box_agreement_rate"] != 1.0:
            raise RuntimeError("The fixed 20-image pilot did not reproduce identical parsed boxes.")

    if INFERENCE_STAGE == "full":
        run_session = run_proposal_jobs(
            worker=worker,
            jobs=jobs,
            result_store=result_store,
            maximum_attempts_per_job=2,
            maximum_consecutive_failed_jobs=3,
            progress_callback=record_progress,
        )
    else:
        run_session = {
            "requested_stage": INFERENCE_STAGE,
            "successful_job_count": len(result_store.completed_keys),
        }
except Exception as error:
    run_error = error
    load_monitor_stop.set()

latest_summary = summarize_proposal_rows(result_store.rows)
print(json.dumps(latest_summary, indent=2, sort_keys=True))
print(json.dumps(one_image_summary, indent=2, sort_keys=True))
print(json.dumps(pilot_summary, indent=2, sort_keys=True))
print(json.dumps(repeat_summary, indent=2, sort_keys=True))


## Operational tables, charts, and private overlays

The tables measure completion, proposal counts, runtime, view coverage, retry status, and truncation. The proposal overlays are private diagnostic evidence and are not uploaded to ClearML. They are not quality evidence because no reference boxes are read here.


In [ ]:
REPORT_ROOT = OUTPUT_ROOT / "report"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
complete_output = len(result_store.rows) == 400
current_job_keys = {
    (str(row["opaque_scene_id"]), str(row["view_id"]))
    for row in result_store.rows
}
current_jobs = [job for job in jobs if job.key in current_job_keys]
expected_stage_jobs = {
    "one_image": 1,
    "pilot": 20,
    "full": 400,
}[INFERENCE_STAGE]
proposal_evidence = (
    collect_proposal_evidence(
        result_store.rows,
        current_jobs,
        history_rows=result_store.history_rows,
        expected_job_count=expected_stage_jobs,
        expected_scene_count=1 if INFERENCE_STAGE == "one_image" else 20,
        expected_view_count=20 if INFERENCE_STAGE == "full" else 1,
    )
    if result_store.rows
    else None
)

if proposal_evidence is not None:
    gate_rows = proposal_evidence["gate_rows"]
    if INFERENCE_STAGE == "pilot":
        gate_rows.append(
            {
                "gate": "repeat agreement",
                "status": "pass" if repeat_summary.get("box_agreement_rate") == 1.0 else "fail",
                "observed": repeat_summary.get("box_agreement_rate", 0.0),
                "expected": 1.0,
            }
        )
    view_rows = proposal_evidence["view_rows"]
    scene_rows = proposal_evidence["scene_rows"]
    failure_rows = proposal_evidence["failure_rows"] or [{"error_type": "none", "count": 0}]
    proposal_counts = proposal_evidence["proposal_counts"]
    runtime_seconds = proposal_evidence["runtime_seconds"]
    response_token_counts = proposal_evidence["response_token_counts"]
    box_area_fractions = proposal_evidence["box_area_fractions"]
    box_aspect_ratios = proposal_evidence["box_aspect_ratios"]
    operational_rows = proposal_evidence["operational_rows"]
    attempt_rows = proposal_evidence["attempt_rows"]
    geometry_summary = proposal_evidence["geometry_summary"]
    outlier_rows = proposal_evidence["outlier_rows"]
    overlay_items = proposal_evidence["overlay_items"]
    outlier_overlay_items = proposal_evidence["outlier_overlay_items"]
else:
    gate_rows = [
        {
            "gate": "requested-stage jobs",
            "status": "pass" if latest_summary["job_count"] == expected_stage_jobs else "fail",
            "observed": latest_summary["job_count"],
            "expected": expected_stage_jobs,
        },
        {
            "gate": "successful jobs",
            "status": "pass" if latest_summary["successful_job_count"] == expected_stage_jobs else "fail",
            "observed": latest_summary["successful_job_count"],
            "expected": expected_stage_jobs,
        },
        {
            "gate": "failed jobs",
            "status": "pass" if latest_summary["failed_job_count"] == 0 else "fail",
            "observed": latest_summary["failed_job_count"],
            "expected": 0,
        },
        {
            "gate": "truncated responses",
            "status": "pass" if latest_summary["possibly_truncated_count"] == 0 else "fail",
            "observed": latest_summary["possibly_truncated_count"],
            "expected": 0,
        },
    ]
    view_rows = proposal_evidence["view_rows"] if proposal_evidence else []
    scene_rows = proposal_evidence["scene_rows"] if proposal_evidence else []
    failure_rows = (
        proposal_evidence["failure_rows"]
        if proposal_evidence and proposal_evidence["failure_rows"]
        else [
            {
                "error_type": type(run_error).__name__ if run_error else "none",
                "count": 1 if run_error else 0,
            }
        ]
    )
    proposal_counts = proposal_evidence["proposal_counts"] if proposal_evidence else []
    runtime_seconds = proposal_evidence["runtime_seconds"] if proposal_evidence else []
    response_token_counts = proposal_evidence["response_token_counts"] if proposal_evidence else []
    box_area_fractions = proposal_evidence["box_area_fractions"] if proposal_evidence else []
    box_aspect_ratios = proposal_evidence["box_aspect_ratios"] if proposal_evidence else []
    operational_rows = proposal_evidence["operational_rows"] if proposal_evidence else []
    attempt_rows = proposal_evidence["attempt_rows"] if proposal_evidence else []
    geometry_summary = proposal_evidence["geometry_summary"] if proposal_evidence else {}
    outlier_rows = proposal_evidence["outlier_rows"] if proposal_evidence else []
    overlay_items = proposal_evidence["overlay_items"] if proposal_evidence else []
    outlier_overlay_items = (
        proposal_evidence["outlier_overlay_items"] if proposal_evidence else []
    )

gate_columns = ["gate", "status", "observed", "expected"]
view_columns = ["view_id", "image_count", "successful_count", "failed_count", "mean_proposal_count", "mean_runtime_seconds", "truncated_count"]
scene_columns = ["scene_index", "image_count", "successful_count", "failed_count", "mean_proposal_count", "mean_runtime_seconds", "truncated_count"]
write_csv_rows(REPORT_ROOT / "completion_gates.csv", gate_columns, gate_rows)
if view_rows and RUN_MODE == "model_inference":
    write_csv_rows(REPORT_ROOT / "proposal_summary_by_view.csv", view_columns, view_rows)
if scene_rows and RUN_MODE == "model_inference":
    write_csv_rows(REPORT_ROOT / "proposal_summary_by_scene.csv", scene_columns, scene_rows)
if view_rows and RUN_MODE == "contract_smoke":
    write_csv_rows(
        REPORT_ROOT / "contract_coverage_by_view.csv",
        ["view_id", "image_count", "successful_count", "failed_count", "truncated_count"],
        view_rows,
    )
if attempt_rows:
    write_csv_rows(
        REPORT_ROOT / "attempt_summary.csv",
        ["attempt_count", "image_count"],
        attempt_rows,
    )
if operational_rows and RUN_MODE == "model_inference":
    write_csv_rows(
        REPORT_ROOT / "operational_results.csv",
        [
            "job_index", "view_id", "status", "attempt_count",
            "proposal_count", "response_token_count", "truncation_check", "runtime_seconds",
            "mean_box_area_fraction",
        ],
        operational_rows,
    )
if outlier_rows and RUN_MODE == "model_inference":
    write_csv_rows(
        REPORT_ROOT / "operational_outliers.csv",
        [
            "reason", "job_index", "view_id", "status", "attempt_count",
            "proposal_count", "response_token_count", "truncation_check", "runtime_seconds",
            "mean_box_area_fraction",
        ],
        outlier_rows,
    )
write_csv_rows(REPORT_ROOT / "failure_summary.csv", ["error_type", "count"], failure_rows)
write_csv_rows(
    REPORT_ROOT / "progress_checkpoints.csv",
    ["session_completed", "session_total", "overall_successful", "latest_status"],
    progress_rows,
)
(REPORT_ROOT / "one_image_preflight.json").write_text(
    json.dumps(one_image_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(REPORT_ROOT / "pilot_summary.json").write_text(
    json.dumps(
        {
            **pilot_summary,
            "estimated_full_run_seconds": estimated_full_run_seconds,
        },
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)
if geometry_summary and RUN_MODE == "model_inference":
    (REPORT_ROOT / "box_geometry_summary.json").write_text(
        json.dumps(geometry_summary, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

proposal_chart = REPORT_ROOT / "proposal_count_distribution.png"
runtime_chart = REPORT_ROOT / "runtime_distribution.png"
view_chart = REPORT_ROOT / "mean_proposals_by_view.png"
token_chart = REPORT_ROOT / "response_token_distribution.png"
area_chart = REPORT_ROOT / "box_area_fraction_distribution.png"
aspect_chart = REPORT_ROOT / "box_aspect_ratio_distribution.png"
runtime_scatter = REPORT_ROOT / "runtime_by_proposal_count.png"
contract_coverage_chart = REPORT_ROOT / "contract_job_coverage_by_view.png"
overlay_grid = REPORT_ROOT / (
    "private_model_proposal_overlays.png"
    if RUN_MODE == "model_inference"
    else "private_contract_input_grid.png"
)
outlier_grid = REPORT_ROOT / "private_model_outlier_overlays.png"
if RUN_MODE == "model_inference" and proposal_counts:
    save_histogram(
        proposal_counts,
        proposal_chart,
        title="LocateAnything proposal count distribution",
        x_label="Proposals per image",
        bins=10,
    )
if RUN_MODE == "model_inference" and runtime_seconds:
    save_histogram(
        runtime_seconds,
        runtime_chart,
        title="LocateAnything runtime distribution",
        x_label="Seconds per image",
        bins=12,
    )
if RUN_MODE == "model_inference" and response_token_counts:
    save_histogram(
        response_token_counts,
        token_chart,
        title="LocateAnything response-token distribution",
        x_label="Response tokens",
        bins=12,
    )
if RUN_MODE == "model_inference" and box_area_fractions:
    save_histogram(
        box_area_fractions,
        area_chart,
        title="LocateAnything normalized box-area distribution",
        x_label="Box area as a fraction of image area",
        bins=12,
    )
if RUN_MODE == "model_inference" and box_aspect_ratios:
    save_histogram(
        box_aspect_ratios,
        aspect_chart,
        title="LocateAnything box aspect-ratio distribution",
        x_label="Box width divided by height",
        bins=12,
    )
if RUN_MODE == "model_inference" and proposal_counts and runtime_seconds:
    save_scatter_plot(
        proposal_counts,
        runtime_seconds,
        runtime_scatter,
        title="LocateAnything runtime by proposal count",
        x_label="Proposals per image",
        y_label="Seconds per image",
    )
if RUN_MODE == "model_inference" and view_rows:
    save_bar_chart(
        [row["view_id"] for row in view_rows],
        [row["mean_proposal_count"] for row in view_rows],
        view_chart,
        title="LocateAnything mean proposal count by view",
        y_label="Mean proposals",
    )
if RUN_MODE == "contract_smoke" and view_rows:
    save_bar_chart(
        [row["view_id"] for row in view_rows],
        [row["successful_count"] for row in view_rows],
        contract_coverage_chart,
        title="Contract-smoke successful jobs by view",
        y_label="Successful jobs",
    )
display_overlay_items = (
    overlay_items
    if RUN_MODE == "model_inference"
    else [{**item, "boxes": []} for item in overlay_items]
)
if display_overlay_items:
    save_image_grid(
        display_overlay_items,
        overlay_grid,
        title=(
            "Private LocateAnything proposal examples"
            if RUN_MODE == "model_inference"
            else "Contract-smoke inputs, no model output"
        ),
        columns=5,
    )
if RUN_MODE == "model_inference" and outlier_overlay_items:
    save_image_grid(
        outlier_overlay_items,
        outlier_grid,
        title="Private LocateAnything operational outliers",
        columns=5,
    )

display_rows(gate_rows, gate_columns)
if view_rows:
    display_rows(
        view_rows,
        view_columns if RUN_MODE == "model_inference" else [
            "view_id", "image_count", "successful_count",
            "failed_count", "truncated_count",
        ],
    )
if attempt_rows:
    display_rows(attempt_rows, ["attempt_count", "image_count"])
display_rows(failure_rows, ["error_type", "count"])
for figure_path in (
    proposal_chart, runtime_chart, view_chart, token_chart, area_chart,
    aspect_chart, runtime_scatter, contract_coverage_chart, overlay_grid,
    outlier_grid,
):
    if figure_path.is_file():
        display(Image.open(figure_path))


## Run record, artifact hashes, and ClearML evidence

ClearML receives configuration, environment details, progress, completion tables, failure counts, aggregate charts, runtime and memory scalars, and hash summaries. Raw responses, per-image boxes, private overlays, and model-input images remain only in the private output bundle.


In [ ]:
peak_gpu_memory_gib = (
    round(torch.cuda.max_memory_allocated() / (1024 ** 3), 4)
    if RUN_MODE == "model_inference" else 0.0
)
peak_gpu_reserved_gib = (
    round(torch.cuda.max_memory_reserved() / (1024 ** 3), 4)
    if RUN_MODE == "model_inference" else 0.0
)
operational_gate_passed = (
    run_error is None
    and INFERENCE_STAGE == "full"
    and latest_summary["job_count"] == 400
    and latest_summary["successful_job_count"] == 400
    and latest_summary["failed_job_count"] == 0
    and latest_summary["possibly_truncated_count"] == 0
    and bool(one_image_summary)
    and one_image_summary.get("status") == "ok"
    and pilot_summary.get("successful_job_count") == 20
    and repeat_summary.get("box_agreement_rate") == 1.0
    and all(row["status"] == "pass" for row in gate_rows)
)
diagnostic_gate_passed = (
    run_error is None
    and INFERENCE_STAGE in {"one_image", "pilot"}
    and bool(one_image_summary)
    and one_image_summary.get("status") == "ok"
    and all(row["status"] == "pass" for row in gate_rows)
    and (
        INFERENCE_STAGE != "pilot"
        or (
            pilot_summary.get("successful_job_count") == 20
            and repeat_summary.get("box_agreement_rate") == 1.0
        )
    )
)
requested_stage_gate_passed = operational_gate_passed or diagnostic_gate_passed
safe_summary = (
    latest_summary
    if RUN_MODE == "model_inference"
    else {
        key: latest_summary[key]
        for key in (
            "job_count", "successful_job_count", "failed_job_count",
            "scene_count", "possibly_truncated_count",
            "configuration_sha256",
        )
    }
)
summary_path = REPORT_ROOT / (
    "proposal_summary.json"
    if RUN_MODE == "model_inference"
    else "contract_summary.json"
)
summary_path.write_text(
    json.dumps(safe_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(REPORT_ROOT / "repeat_agreement.json").write_text(
    json.dumps(repeat_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
tracker = ExperimentTracker(mode="disabled")
tracking_error = None
try:
    print("Tracking checkpoint: starting ClearML after inference", flush=True)
    tracker = ExperimentTracker.start(
        project_name="Visual Peer Inspector/02 Detection and Segmentation",
        task_name=f"ds0-locateanything-{EXECUTION_ENVIRONMENT}-candidate",
        task_type="inference",
        properties=tracking_properties,
        configuration=tracking_configuration,
    )
    tracker.add_tags([
        "candidate", "notebook-04", EXECUTION_ENVIRONMENT,
        RUN_MODE, INFERENCE_STAGE, "smoke-20",
    ])
    print(
        f"Tracking checkpoint: task ready ({tracker.task_id or 'tracking disabled'})",
        flush=True,
    )
    scalar_values = {
        "completed jobs": latest_summary["successful_job_count"],
        "failed jobs": latest_summary["failed_job_count"],
        "truncated responses": latest_summary["possibly_truncated_count"],
        "repeat agreement": repeat_summary["box_agreement_rate"],
        "model download seconds": model_download_seconds,
        "model load seconds": model_load_seconds,
        "estimated full run seconds": estimated_full_run_seconds,
    }
    if RUN_MODE == "model_inference":
        scalar_values.update(
            {
                "mean proposals per image": latest_summary["mean_proposals_per_successful_image"],
                "mean seconds per image": latest_summary["mean_runtime_seconds"],
                "p95 seconds per image": latest_summary["p95_runtime_seconds"],
                "peak GPU allocated GiB": peak_gpu_memory_gib,
                "peak GPU reserved GiB": peak_gpu_reserved_gib,
                "retried images": geometry_summary.get("retried_image_count", 0),
                "invalid boxes": geometry_summary.get("invalid_box_count", 0),
                "duplicate boxes": geometry_summary.get("duplicate_box_count", 0),
            }
        )
    for series, value in scalar_values.items():
        tracker.report_scalar(title="DS0 proposals", series=series, value=value)
    tracker.report_table(
        title="DS0 proposals",
        series="completion gates",
        columns=gate_columns,
        rows=gate_rows,
    )
    if view_rows:
        tracker.report_table(
            title="DS0 proposals",
            series="summary by view",
            columns=(
                view_columns
                if RUN_MODE == "model_inference"
                else [
                    "view_id", "image_count", "successful_count",
                    "failed_count", "truncated_count",
                ]
            ),
            rows=view_rows,
        )
    if attempt_rows:
        tracker.report_table(
            title="DS0 proposals",
            series="attempt summary",
            columns=["attempt_count", "image_count"],
            rows=attempt_rows,
        )
    tracker.report_table(
        title="DS0 proposals",
        series="failure summary",
        columns=["error_type", "count"],
        rows=failure_rows,
    )
    for series, chart_path in (
        ("proposal count distribution", proposal_chart),
        ("runtime distribution", runtime_chart),
        ("mean proposals by view", view_chart),
        ("response token distribution", token_chart),
        ("box area distribution", area_chart),
        ("box aspect ratio distribution", aspect_chart),
        ("runtime by proposal count", runtime_scatter),
        ("contract coverage by view", contract_coverage_chart),
    ):
        if chart_path.is_file():
            tracker.report_image_file(
                title="DS0 proposals",
                series=series,
                path=chart_path,
                evidence_kind="aggregate_chart",
            )
    tracker.upload_summary("run summary", safe_summary)
    tracker.upload_summary("one-image preflight", one_image_summary)
    tracker.upload_summary(
        "20-image pilot",
        {
            **pilot_summary,
            "estimated_full_run_seconds": estimated_full_run_seconds,
        },
    )
    tracker.upload_summary("repeat agreement", repeat_summary)
    tracker.upload_summary(
        "model load checkpoints",
        {"checkpoints": model_load_rows},
    )
    tracker.upload_summary("GPU environment", gpu_evidence)
    if RUN_MODE == "model_inference":
        tracker.upload_summary("box geometry", geometry_summary)
    preclose_artifacts = {
        path.relative_to(OUTPUT_ROOT).as_posix(): file_sha256(path)
        for path in sorted(REPORT_ROOT.rglob("*"))
        if path.is_file() and path.name != "artifact_manifest.json"
    }
    tracker.upload_summary("pre-close artifact hashes", preclose_artifacts)
    if not requested_stage_gate_passed:
        tracker.mark_failed(type(run_error).__name__ if run_error else "IncompleteRun")
except Exception as error:
    tracking_error = error
    try:
        tracker.mark_failed(type(error).__name__)
    except Exception:
        pass
finally:
    try:
        tracker.close()
    except Exception as error:
        if tracking_error is None:
            tracking_error = error

if operational_gate_passed and tracking_error is None:
    final_status = "completed"
elif diagnostic_gate_passed and tracking_error is None:
    final_status = "diagnostic_completed"
else:
    final_status = "failed"
run_record = {
    "schema_version": 3,
    "stage": "ds0_locateanything_proposals",
    "status": final_status,
    "tracking_status": "completed" if tracking_error is None else "failed",
    "execution_environment": EXECUTION_ENVIRONMENT,
    "run_mode": RUN_MODE,
    "inference_stage": INFERENCE_STAGE,
    "producer_kind": producer_kind,
    "notebook_name": NOTEBOOK_NAME,
    "provenance": PROVENANCE,
    "kaggle_notebook_slug": "esreekarreddy/toysad-8k-ds0-locateanything-proposals",
    "platform_versions": PLATFORM_VERSIONS,
    "clearml_task_id": tracker.task_id,
    "input_manifest_sha256": package_audit.manifest_sha256,
    "configuration": tracking_configuration,
    "configuration_sha256": configuration.sha256,
    "gpu": gpu_evidence,
    "model_load_seconds": round(model_load_seconds, 4),
    "model_download_seconds": round(model_download_seconds, 4),
    "peak_gpu_allocated_gib": peak_gpu_memory_gib,
    "peak_gpu_reserved_gib": peak_gpu_reserved_gib,
    "one_image_preflight": one_image_summary,
    "pilot_summary": pilot_summary,
    "estimated_full_run_seconds": estimated_full_run_seconds,
    "repeat_agreement": repeat_summary,
    "run_session": run_session,
    "summary": safe_summary,
    "error_type": (
        type(tracking_error).__name__
        if tracking_error is not None
        else type(run_error).__name__ if run_error is not None else None
    ),
}
run_record_path = REPORT_ROOT / "run_record.json"
run_record_temporary = run_record_path.with_suffix(".json.incomplete")
run_record_temporary.write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
os.replace(run_record_temporary, run_record_path)

artifact_paths = [OUTPUT_ROOT / "proposal_results.jsonl"]
if repeat_results_path.is_file():
    artifact_paths.append(repeat_results_path)
artifact_paths.extend(
    path for path in sorted(REPORT_ROOT.rglob("*"))
    if path.is_file() and path.name != "artifact_manifest.json"
)
artifact_manifest = {
    path.relative_to(OUTPUT_ROOT).as_posix(): file_sha256(path)
    for path in artifact_paths
}
artifact_manifest_path = REPORT_ROOT / "artifact_manifest.json"
artifact_manifest_path.write_text(
    json.dumps(artifact_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
from visual_peer_inspector.verification import verify_artifact_manifest

artifact_verification = verify_artifact_manifest(OUTPUT_ROOT)

print(f"Run mode: {RUN_MODE}")
print(f"Run status: {final_status}")
print(f"ClearML task ID: {run_record['clearml_task_id'] or 'tracking disabled'}")
print(f"Verified artifact count: {artifact_verification['artifact_count']}")
if tracking_error is not None:
    raise RuntimeError("ClearML evidence logging failed.") from tracking_error
if run_error is not None:
    raise RuntimeError("Proposal run failed after preserving partial evidence.") from run_error
if final_status == "failed":
    raise RuntimeError("Proposal run did not pass its completion gates.")
